# Hair Shifter — video inference on Colab

Re-renders a video with the hairstyle from a reference photo, using `src/inference.py`.

**Runtime → Change runtime type → GPU** before running.

Run the cells top to bottom. The last setup cell launches a Gradio UI where you pick the video, the reference image, the checkpoint and the inference options. Defaults are the sample inputs in `assets/test_images/` (`phuc.mp4` + `ken.png`).

Why a subprocess: the project needs Python 3.10 (`pyproject.toml`), Colab runs a newer Python. The model runs in a 3.10 Poetry venv; the UI runs in the notebook kernel.

In [ ]:
#@title 1. Clone the repo { display-mode: "form" }
REPO_URL = "https://github.com/DragMaid/DP-AR-Hair.git"  #@param {type:"string"}
BRANCH = "feat/stability"  #@param {type:"string"}
REPO_DIR = "/content/DP-AR-Hair"

import os
if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch && git -C {REPO_DIR} checkout {BRANCH} && git -C {REPO_DIR} pull
%cd {REPO_DIR}

In [ ]:
#@title 2. Install tools (ninja, uv, poetry, gradio) { display-mode: "form" }
!command -v ninja >/dev/null 2>&1 || ( \
  wget -q https://github.com/ninja-build/ninja/releases/download/v1.8.2/ninja-linux.zip -O /tmp/ninja.zip && \
  sudo unzip -o /tmp/ninja.zip -d /usr/local/bin/ && \
  sudo update-alternatives --install /usr/bin/ninja ninja /usr/local/bin/ninja 1 --force )
!pip install -q uv poetry gradio

In [ ]:
#@title 3. Build the Python 3.10 environment (takes a while) { display-mode: "form" }
!uv python install 3.10
!poetry config virtualenvs.in-project true
!poetry env use "$(uv python find 3.10)"
!poetry install --with vision,full --no-interaction

VENV_PYTHON = f"{REPO_DIR}/.venv/bin/python"
!{VENV_PYTHON} -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
#@title 4. (Optional) Mount Google Drive for checkpoints { display-mode: "form" }
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
#@title 5. Launch the Gradio UI { display-mode: "form" }
import os
import subprocess
from pathlib import Path
import gradio as gr

REPO_DIR = "/content/DP-AR-Hair"
VENV_PYTHON = f"{REPO_DIR}/.venv/bin/python"
OUT_DIR = Path(REPO_DIR) / "outputs"
OUT_DIR.mkdir(exist_ok=True)

DEFAULT_VIDEO = f"{REPO_DIR}/assets/test_images/phuc.mp4"
DEFAULT_REFERENCE = f"{REPO_DIR}/assets/test_images/ken.png"
DEFAULT_CHECKPOINT = "/content/drive/MyDrive/checkpoints/epoch_0100.pt"  #@param {type:"string"}


def build_cmd(video, reference, checkpoint, output, batch_size, ema, align, poor):
    cmd = [VENV_PYTHON, "src/inference.py",
           "--video", video, "--reference", reference,
           "--checkpoint", checkpoint, "--output", output,
           "--batch_size", str(int(batch_size))]
    if ema:
        cmd.append("--ema")
    if align:
        cmd.append("--align")
    if poor:
        cmd.append("--poor")
    return cmd


def run(video, reference, checkpoint_path, checkpoint_file,
        batch_size, ema, align, poor):
    # An uploaded file wins over the path box
    checkpoint = checkpoint_file or checkpoint_path
    for label, path in [("Video", video), ("Reference image", reference),
                        ("Checkpoint", checkpoint)]:
        if not path or not Path(path).exists():
            raise gr.Error(f"{label} not found: {path!r}")

    avi = str(OUT_DIR / "result.avi")
    mp4 = str(OUT_DIR / "result.mp4")
    cmd = build_cmd(video, reference, checkpoint, avi,
                    batch_size, ema, align, poor)

    log = "$ " + " ".join(cmd) + "\n"
    yield log, None
    env = {**os.environ, "PYTHONPATH": "src:libs"}
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        log += line
        yield log, None
    if proc.wait() != 0:
        raise gr.Error(f"Inference failed (exit {proc.returncode}); see the log.")

    # MJPG .avi does not play in browsers; re-encode to H.264 mp4
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", avi,
                    "-c:v", "libx264", "-pix_fmt", "yuv420p", mp4], check=True)
    yield log + f"\nSaved: {avi}\nPreview: {mp4}\n", mp4


with gr.Blocks(title="Hair Shifter") as demo:
    gr.Markdown("## Hair Shifter — video hair transfer")
    with gr.Row():
        with gr.Column():
            video = gr.Video(value=DEFAULT_VIDEO, label="Driving video")
            reference = gr.Image(value=DEFAULT_REFERENCE, type="filepath",
                                 label="Hair reference image")
            checkpoint_path = gr.Textbox(value=DEFAULT_CHECKPOINT,
                                         label="Checkpoint path (.pt)")
            checkpoint_file = gr.File(label="...or upload a checkpoint",
                                      file_types=[".pt", ".pth"], type="filepath")
            batch_size = gr.Slider(1, 8, value=2, step=1, label="Batch size")
            ema = gr.Checkbox(False, label="Use EMA weights (only after ~10k+ training steps)")
            align = gr.Checkbox(False, label="Align every frame")
            poor = gr.Checkbox(True, label="Poor mode (free HairFastGAN after anchor; saves VRAM)")
            go = gr.Button("Run", variant="primary")
        with gr.Column():
            result = gr.Video(label="Result")
            log = gr.Textbox(label="Log", lines=20, max_lines=20, autoscroll=True)

    go.click(run,
             inputs=[video, reference, checkpoint_path, checkpoint_file,
                     batch_size, ema, align, poor],
             outputs=[log, result])

demo.queue().launch(share=True, debug=True)

In [ ]:
#@title (Fallback) Run without the UI { display-mode: "form" }
VIDEO = "/content/DP-AR-Hair/assets/test_images/phuc.mp4"  #@param {type:"string"}
REFERENCE = "/content/DP-AR-Hair/assets/test_images/ken.png"  #@param {type:"string"}
CHECKPOINT = "/content/drive/MyDrive/checkpoints/epoch_0100.pt"  #@param {type:"string"}
OUTPUT = "/content/DP-AR-Hair/outputs/result.avi"  #@param {type:"string"}
BATCH_SIZE = 2  #@param {type:"integer"}
EMA = False  #@param {type:"boolean"}
ALIGN = False  #@param {type:"boolean"}
POOR = True  #@param {type:"boolean"}

flags = " ".join(f for f, on in [("--ema", EMA), ("--align", ALIGN), ("--poor", POOR)] if on)
!mkdir -p "$(dirname {OUTPUT})"
!cd /content/DP-AR-Hair && PYTHONPATH=src:libs .venv/bin/python src/inference.py \
  --video "{VIDEO}" --reference "{REFERENCE}" --checkpoint "{CHECKPOINT}" \
  --output "{OUTPUT}" --batch_size {BATCH_SIZE} {flags}
!ffmpeg -y -loglevel error -i "{OUTPUT}" -c:v libx264 -pix_fmt yuv420p "{OUTPUT}.mp4"

from IPython.display import Video
Video(OUTPUT + ".mp4", embed=True, width=512)